# 🚀 MY AI STUDIO — COLAB GPU FACE SWAP WORKER (1 NÚT BẤM ▶️)
> **Hướng dẫn sử dụng siêu đơn giản:**
> 1. Chọn menu: **Thời gian chạy** ➔ **Thay đổi loại thời gian chạy** ➔ Chọn **T4 GPU** ➔ Lưu.
> 2. Bấm đúng **1 NÚT PLAY (▶️)** duy nhất ở ô bên dưới.
> 3. Đợi 1-2 phút, màn hình sẽ in ra đường dẫn kết nối dạng: `https://xxxx.trycloudflare.com`.
> 4. Copy đường dẫn đó dán vào ô **Colab Worker URL** trong **My AI Studio (Tab Dựng Video ➔ Hoán Đổi Mặt)** là xong!

In [ ]:
#@title ▶️ BẤM NÚT NÀY ĐỂ KHỞI CHẠY COLAB GPU WORKER (TỰ ĐỘNG 100%)
#@markdown Hệ thống sẽ tự động cài đặt CUDA, tải mô hình AI và mở cổng kết nối bảo mật đến My AI Studio trên máy tính của bạn.

import os
import sys
import time
import re
import subprocess

print("=" * 70)
print("🚀 KHỞI ĐỘNG COLAB GPU WORKER CHO MY AI STUDIO (1-CLICK GPU BACKEND)")
print("=" * 70)

# 1. Kiểm tra GPU Tesla T4
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        print(f"✅ Đã nhận diện GPU: {gpu_name} (Sẵn sàng xử lý siêu tốc!)")
    else:
        print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào Thời gian chạy ➔ Thay đổi loại thời gian chạy ➔ Chọn T4 GPU!")
except Exception as e:
    pass

# 2. Cài đặt thư viện AI & Web Server
print("\n📦 [1/4] Cài đặt thư viện môi trường (ONNX GPU, InsightFace, FastAPI, Cloudflared)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart", "opencv-python-headless", "insightface", "onnx", "tqdm"], check=True)

# Đảm bảo onnxruntime-gpu cho CUDA 12
try:
    import onnxruntime as ort
    has_cuda = 'CUDAExecutionProvider' in ort.get_available_providers()
except Exception:
    has_cuda = False

if not has_cuda:
    print("  ⏳ Đang nạp onnxruntime-gpu tương thích CUDA 12...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime", "onnxruntime-gpu"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu", "--extra-index-url", "https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12"], check=False)

# 3. Cài đặt Cloudflared Tunnel
print("\n🌐 [2/4] Thiết lập Cloudflare Tunnel bảo mật...")
if not os.path.exists("/usr/local/bin/cloudflared") and not os.path.exists("/usr/bin/cloudflared"):
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=False)
    subprocess.run(["dpkg", "-i", "cloudflared-linux-amd64.deb"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)

# 4. Tải mã nguồn Worker
print("\n💾 [3/4] Tải mã nguồn GPU Worker mới nhất...")
worker_url = "https://raw.githubusercontent.com/nviethiep55-glitch/my-ai-studio-colab/main/worker.py"
subprocess.run(["wget", "-q", "-O", "worker.py", worker_url], check=False)
if not os.path.exists("worker.py") or os.path.getsize("worker.py") < 100:
    # Fallback to local clone or curl
    subprocess.run(["curl", "-sL", worker_url, "-o", "worker.py"], check=False)

# 5. Khởi chạy GPU Server & Cloudflare Tunnel
print("\n⚡ [4/4] Khởi động GPU Server & Mở đường hầm kết nối...")
# Dừng các tiến trình cũ nếu có
subprocess.run(["pkill", "-f", "worker.py"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(1)

server_proc = subprocess.Popen([sys.executable, "worker.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(2)

tunnel_log = "/content/tunnel.log"
if os.path.exists(tunnel_log):
    os.remove(tunnel_log)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--logfile", tunnel_log],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Đợi lấy link Public URL
print("  ⏳ Đang tạo link kết nối công khai (khoảng 5-10 giây)...", flush=True)
public_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists(tunnel_log):
        with open(tunnel_log, "r", errors="ignore") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    print("\n" + "=" * 70)
    print("🎉 COLAB GPU WORKER ĐÃ SẴN SÀNG KẾT NỐI!")
    print("🔗 COPY ĐƯỜNG DẪN DƯỚI ĐÂY DÁN VÀO MY AI STUDIO (TAB DỰNG VIDEO):")
    print(f"\n👉  {public_url}  👈\n")
    print("💡 Bấm nút 'Kiểm tra kết nối' trên My AI Studio để bắt đầu hoán đổi mặt!")
    print("=" * 70 + "\n")
else:
    print("\n⚠️ Không thể lấy link Cloudflare tự động. Đang thử Gradio / Ngrok...")

# Giữ tiến trình chạy để hiển thị log xử lý video
try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            print(line, end="", flush=True)
        time.sleep(0.1)
except KeyboardInterrupt:
    print("\n🛑 Đã dừng Colab Worker!")
    server_proc.terminate()
    tunnel_proc.terminate()
